In [ ]:
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.2,
    callbacks=[lf_cb],
)


In [ ]:

import os


# ---- Langfuse (self-hosted) ----
os.environ.setdefault("LANGFUSE_PUBLIC_KEY",  "")
os.environ.setdefault("LANGFUSE_SECRET_KEY",  "")
# If you run Langfuse locally via docker compose the UI/API is on :3000 by default:
os.environ.setdefault("LANGFUSE_HOST",        "http://localhost:3000")

# ---- OpenAI ----
# Works with the standard OpenAI SDK interface but routed through Langfuse integration
os.environ.setdefault("OPENAI_API_KEY",       "")  # or use environment secrets

for k in ["OPENAI_API_KEY","LANGFUSE_PUBLIC_KEY","LANGFUSE_SECRET_KEY","LANGFUSE_BASE_URL"]:
    print(k, "=", ("set" if os.getenv(k) else "MISSING"))


OPENAI_API_KEY = set
LANGFUSE_PUBLIC_KEY = set
LANGFUSE_SECRET_KEY = set
LANGFUSE_BASE_URL = MISSING


In [6]:
# --- helper: fill {{var}} or {var} placeholders manually ---
def substitute_placeholders(messages_or_text, vars_):
    if isinstance(messages_or_text, list):  # chat messages
        new_msgs = []
        for m in messages_or_text:
            c = m["content"]
            for k, v in vars_.items():
                c = c.replace(f"{{{{{k}}}}}", str(v)).replace(f"{{{k}}}", str(v))
            new_msgs.append({**m, "content": c})
        return new_msgs
    else:  # single text prompt
        c = messages_or_text
        for k, v in vars_.items():
            c = c.replace(f"{{{{{k}}}}}", str(v)).replace(f"{{{k}}}", str(v))
        return c


In [7]:
# -------------------------------
# 0) Imports & shared utilities
# -------------------------------
import uuid
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

DEBUG_PREVIEW = True  # set False to disable preview prints

def run_prompt(prompt_name: str, label: str, variables: dict, model: ChatOpenAI, lf_cb: CallbackHandler):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    # Compile with vars (Langfuse), then ensure placeholders are actually substituted
    compiled = prompt.compile(variables=variables)          # may return list (chat) or str (text)
    compiled = substitute_placeholders(compiled, variables) # guarantee {{var}}/{var} filled

    # Optional preview for quick verification (no secrets)
    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(f"{m.get('role','?')}: {str(m.get('content',''))[:240]}" for m in compiled)
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {"prompt_version": getattr(prompt, "version", None)},
            }
        },
    }

    resp = model.invoke(compiled, config=config)
    print(f"\n=== {prompt_name}:{label} ===")
    print(resp.content)
    lf_client.flush()



In [8]:
# ==========================================
# 1) Common setup (Langfuse + OpenAI client)
# ==========================================
lf_client = get_client()
lf_cb = CallbackHandler()  # Langfuse v3.8.1: no kwargs
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, callbacks=[lf_cb])
parser = StrOutputParser()  # (kept for symmetry; not explicitly used here)


# =======================
# 2) Example 1: Summarizer
# =======================
summarizer_vars = {
    "source_text": """
Acme shipped Order #48291 in two parts due to inventory constraints. 
Customer claims double charge on 10/03 and requests refund for one transaction. 
Support ticket escalated to billing; resolution ETA 48 hours. Next meeting Friday 2 PM.
""",
    "tone": "neutral",
    "length": "5–7 sentences",
    "bullet_points": "yes",
}
run_prompt("demo-summarizer", "production", summarizer_vars, model, lf_cb)


--- PREVIEW: demo-summarizer:production ---
system: You are a precise summarizer. Write in a neutral tone.
Target length: 5–7 sentences. 
If yes == "yes", return a concise bulleted list. 
Preserve key facts, dates, figures, and decisions. Avoid fluff.

user: Summarize this text:
---

Acme shipped Order #48291 in two parts due to inventory constraints. 
Customer claims double charge on 10/03 and requests refund for one transaction. 
Support ticket escalated to billing; resolution ETA 48 hours. N
--- END PREVIEW ---

=== demo-summarizer:production ===
- Acme shipped Order #48291 in two parts due to inventory constraints.
- Customer reported a double charge on 10/03 and requested a refund for one transaction.
- The support ticket has been escalated to billing.
- Estimated time for resolution is 48 hours.
- Next meeting scheduled for Friday at 2 PM.


In [ ]:
# ==================================
# 3) Example 2: Conversation chat bot
# ==================================
conv_vars = {
    "brand": "Acme Gadgets",
    "language": "English",
    "knowledge": """Shipping takes 3–5 business days. 
Returns accepted within 30 days in original packaging. 
Warranty: 1 year for manufacturing defects; batteries excluded.""",
    "history_block": """User: Do you ship internationally?
Assistant: We currently ship only within the EU and UK.""",
    "user_input": "My device won’t hold charge. Is the battery covered?",
}
run_prompt("demo-conversation-bot", "production", conv_vars, model, lf_cb)





--- PREVIEW: demo-conversation-bot:production ---
system: You are a helpful, concise support assistant for Acme Gadgets.
Answer ONLY using the "Knowledge" below. If the answer is not covered, say you don't know and offer to escalate.
Respond in English.

Knowledge:
Shipping takes 3–5 business days
assistant: Conversation history so far:
User: Do you ship internationally?
Assistant: We currently ship only within the EU and UK.

user: My device won’t hold charge. Is the battery covered?
--- END PREVIEW ---

=== demo-conversation-bot:production ===
Batteries are excluded from the warranty, so they are not covered. If you have a manufacturing defect with the device itself, it may be covered under the 1-year warranty.


In [ ]:
# ==========================================
# 4) Example 3: Ticket triage (JSON classifier)
# ==========================================
triage_vars = {
    "issue_text": "I was charged twice for the same order yesterday. Please reverse one of the payments.",
    "customer_tier": "Gold",
    "recent_orders": "Order#94821 on 2025-11-02; Visa **** 2219",
}
run_prompt("demo-ticket-triage", "production", triage_vars, model, lf_cb)


--- PREVIEW: demo-ticket-triage:production ---
You are a ticket triage assistant. Read the issue and return STRICT JSON with keys:
- category: one of ["billing","shipping","returns","technical","account","other"]
- priority: one of ["low","medium","high","urgent"]
- route_to: one of ["billing-desk","shipping-desk","tech-desk","returns-desk","account-desk","general-queue"]
- rationale: short reason (max 30 words)

Context:
- Customer tier: Gold
- Recent orders: Order#94821 on 2025-11-02; Visa **** 2219

Issue:
I was charged twice for the same order yesterday. Please reverse one of the payments.

Return ONLY JSON, no extra text.

--- END PREVIEW ---

=== demo-ticket-triage:production ===
```json
{
  "category": "billing",
  "priority": "high",
  "route_to": "billing-desk",
  "rationale": "Customer was charged twice for a single order."
}
```


In [ ]:
!pip install wandb weave

In [ ]:
import uuid
import time
import wandb  # 👈 add this
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import weave

In [ ]:
wandb_run = wandb.init(
    project="llm-prompt-experiments",  # your project name
    name="local-prompt-testing",       # or something dynamic
    config={
        "model": "gpt-4o-mini",
        "environment": "local-dev",
    },
)


In [ ]:
def run_prompt(
    prompt_name: str,
    label: str,
    variables: dict,
    model: ChatOpenAI,
    lf_cb: CallbackHandler,
):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    # Compile with vars (Langfuse), then ensure placeholders are actually substituted
    compiled = prompt.compile(variables=variables)          # may return list (chat) or str (text)
    compiled = substitute_placeholders(compiled, variables) # guarantee {{var}}/{var} filled

    # Optional preview for quick verification (no secrets)
    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(
                f"{m.get('role','?')}: {str(m.get('content',''))[:240]}"
                for m in compiled
            )
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {
                    "prompt_version": getattr(prompt, "version", None),
                },
            }
        },
    }

    # 👉 measure latency around the actual model call
    start_time = time.time()
    resp = model.invoke(compiled, config=config)
    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000.0

    print(f"\n=== {prompt_name}:{label} ===")
    print(resp.content)

    # ---------------------------
    # W&B: log prompt experiment + latency
    # ---------------------------
    # Try to get token usage if available from LangChain/OpenAI
    token_usage = {}
    try:
        # Depending on langchain_openai version, adjust this if needed
        token_usage = resp.response_metadata.get("token_usage", {})
    except Exception:
        token_usage = {}

    wandb.log(
        {
            # latency observability
            "latency_ms": latency_ms,

            # prompt experiment info
            "prompt_name": prompt_name,
            "label": label,
            "prompt_version": getattr(prompt, "version", None),
            "model_name": getattr(model, "model_name", None) or getattr(model, "model", None),

            # tokens (if available)
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),

            # you can also log something derived from variables
            "tone": variables.get("tone"),
            "length": variables.get("length"),
        }
    )

    lf_client.flush()
    return resp


In [ ]:
import uuid
import time
import wandb
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import weave  # optional if using W&B Weave decorators

DEBUG_PREVIEW = True  # toggle preview output

# 👇 Init W&B project and run
wandb_run = wandb.init(
    project="llm-prompt-experiments",
    name="local-prompt-testing",
    config={
        "model": "gpt-4o-mini",
        "environment": "local-dev",
    },
)

def substitute_placeholders(compiled, variables):
    # Simple fallback template substitution
    if isinstance(compiled, str):
        for k, v in variables.items():
            compiled = compiled.replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    elif isinstance(compiled, list):
        for msg in compiled:
            if isinstance(msg.get("content"), str):
                for k, v in variables.items():
                    msg["content"] = msg["content"].replace("{{" + k + "}}", str(v)).replace("{" + k + "}", str(v))
    return compiled

def run_prompt(
    prompt_name: str,
    label: str,
    variables: dict,
    model: ChatOpenAI,
    lf_cb: CallbackHandler,
):
    lf_client = get_client()
    prompt = lf_client.get_prompt(prompt_name, label=label)

    compiled = prompt.compile(variables=variables)
    compiled = substitute_placeholders(compiled, variables)

    if DEBUG_PREVIEW:
        if isinstance(compiled, list):
            preview = "\n".join(
                f"{m.get('role','?')}: {str(m.get('content',''))[:240]}"
                for m in compiled
            )
        else:
            preview = str(compiled)[:600]
        print(f"\n--- PREVIEW: {prompt_name}:{label} ---\n{preview}\n--- END PREVIEW ---")

    trace_id = f"{prompt_name}-{label}-{uuid.uuid4()}"
    config = {
        "callbacks": [lf_cb],
        "langfuse": {
            "trace_args": {
                "id": trace_id,
                "name": f"{prompt_name}:{label}",
                "user_id": "udara-local",
                "tags": [f"prompt:{prompt_name}", f"label:{label}"],
                "metadata": {
                    "prompt_version": getattr(prompt, "version", None),
                },
            }
        },
    }

    # 👇 W&B trace context for trace tree logging (optional)
    with wandb.trace(name=f"{prompt_name}:{label}") as span:
        start_time = time.time()
        resp = model.invoke(compiled, config=config)
        end_time = time.time()
        latency_ms = (end_time - start_time) * 1000.0

        print(f"\n=== {prompt_name}:{label} ===")
        print(resp.content)

        # 👇 Extract token usage if available
        token_usage = {}
        try:
            token_usage = resp.response_metadata.get("token_usage", {})
        except Exception:
            pass

        # 👇 Log prompt experiment + latency
        wandb.log({
            "latency_ms": latency_ms,
            "prompt_name": prompt_name,
            "label": label,
            "prompt_version": getattr(prompt, "version", None),
            "model_name": getattr(model, "model_name", None),
            "prompt_tokens": token_usage.get("prompt_tokens"),
            "completion_tokens": token_usage.get("completion_tokens"),
            "total_tokens": token_usage.get("total_tokens"),
            "tone": variables.get("tone"),
            "length": variables.get("length"),
        })

    lf_client.flush()
    return resp

# 👇 Example prompt variables
triage_vars = {
    "issue_text": "I was charged twice for the same order yesterday. Please reverse one of the payments.",
    "customer_tier": "Gold",
    "recent_orders": "Order#94821 on 2025-11-02; Visa **** 2219",
}

# 👇 You need to define these before calling `run_prompt`
model = ChatOpenAI(model_name="gpt-4o", temperature=0.2)
lf_cb = CallbackHandler()

# 👇 Trigger prompt run
run_prompt("demo-ticket-triage", "production", triage_vars, model, lf_cb)


In [ ]:
https://chatgpt.com/s/dr_691c5accfd0c81919064e18fb830b055